# Väderassistent

## Syfte

Detta projekt är en enkel väderassistent som hämtar aktuell väderdata från ett API.

Användaren skriver in en stad och programmet hämtar stadens koordinater och aktuell temperatur och väderinformation.

In [1]:
import requests
import csv
from datetime import datetime 

## Importera bibliotek

Vi använder requests för att hämta data från väder-API:t.
csv används för att spara väderdata i en CSV-fil.
datetime används för att spara datum och tid.

In [22]:
def get_coordinates(city):
    url = "https://geocoding-api.open-meteo.com/v1/search"

    params = {
        "name": city,
        "count": 1
    }

    try:
        response = requests.get(url, params=params)
        data = response.json()
    except:
        return None, None

    print(data)
    
    latitude = data["results"][0]["latitude"]
    longitude = data["results"][0]["longitude"]

    return latitude, longitude

## Hämta koordinater

Användaren skriver in en stad. Väder-API:et behöver stadens koordinater för att kunna hämta rätt väderdata.

In [23]:
city = input("Vilken stad vill du veta väder för? ")

latitude, longitude = get_coordinates(city)

{'results': [{'id': 2673730, 'name': 'Stockholm', 'latitude': 59.32938, 'longitude': 18.06871, 'elevation': 17.0, 'feature_code': 'PPLC', 'country_code': 'SE', 'admin1_id': 2673722, 'admin2_id': 2673723, 'timezone': 'Europe/Stockholm', 'population': 1515017, 'country_id': 2661886, 'country': 'Sweden', 'admin1': 'Stockholm County', 'admin2': 'Stockholm Municipality'}], 'generationtime_ms': 0.69987774}


In [8]:
weather_url = "https://api.open-meteo.com/v1/forecast"

weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": "temperature_2m,weather_code"
}

## Hämta väderdata

Programmet använder koordinaterna för att hämta aktuell temperatur och väderinformation från väder-API:et.

In [9]:
weather_response = requests.get(
    weather_url,
    params=weather_params,
    timeout=10
)

weather_data = weather_response.json()

In [10]:
temperature = weather_data["current"]["temperature_2m"]
weather_code = weather_data["current"]["weather_code"]

In [11]:
def describe_weather(weather_code):
    if weather_code == 0:
        return "Klart väder"
    elif weather_code in [1, 2, 3]:
        return "Molnigt"
    elif weather_code in [51, 53, 55]:
        return "Duggregn"
    elif weather_code in [61, 63, 65]:
        return "Regn"
    elif weather_code in [71, 73, 75]:
        return "Snö"
    else:
        return "Annat väder"

## Analysera vädret

Programmet tolkar väderkoden från API:et och visar en enkel beskrivning av vädret. Temperaturen analyseras också för att kunna visa en varning vid mycket hög eller låg temperatur.

In [12]:
condition = describe_weather(weather_code)

print(f"Temperatur: {temperature} °C")
print(f"Väder: {condition}")

Temperatur: 14.5 °C
Väder: Klart väder


## Spara väderdata

Väderinformationen sparas i en CSV-fil tillsammans med stad, temperatur, väder och datum och tid.

In [13]:
with open("weather_data.csv", "a", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)

    writer.writerow([
        city,
        temperature,
        condition,
        datetime.now().strftime("%Y-%m-%d %H:%M")
    ])

In [24]:
with open("weather_data.csv", "r", encoding="utf-8") as file:
    data = file.read()

print(data)

Stockholm,16.4,Molnigt,2026-10-02 18:12
Göteborg,16.2,Klart väder,2026-10-02 18:12
stockholm,14.5,Klart väder,2026-10-03 21:24



## Klasser och arv

WeatherData används för att samla information om stad och temperatur.
WeatherAssistant ärver från WeatherData och kan dessutom visa en temperaturvarning.

In [14]:
class WeatherData:
    def __init__(self, city, temperature):
        self.city = city
        self.temperature = temperature

    def show_temperature(self):
        return f"{self.city}: {self.temperature} °C"

In [18]:
def analyze_temperature(temperature):
    if temperature >= 30:
        return "Varning: Det är mycket varmt."
    elif temperature < 0:
        return "Varning: Det är minusgrader."
    else:
        return "Temperaturen är normal."


temperatures = [temperature, 5, 25]

for temp in temperatures:
    print(analyze_temperature(temp))

Temperaturen är normal.
Temperaturen är normal.
Temperaturen är normal.


In [16]:
class WeatherAssistant(WeatherData):
    def show_warning(self):
        return analyze_temperature(self.temperature)

In [17]:
weather = WeatherAssistant(city, temperature)

print(weather.show_warning())

Temperaturen är normal.


## Resultat

Väderassistenten hämtar väderdata för en vald stad, visar temperatur och väderförhållande och sparar informationen i en CSV-fil.